In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Prepare LLM batch inputs
!python /workspace/src/qa_generation/llm_qa_generator_batch.py prepare \
    /workspace/data/processed/canonical_facts/gold.csv \
    /workspace/data/qa/llm_batch/output.jsonl \
    /workspace/data/qa/llm_batch/mapping.csv \
    --model gpt-4o-mini \
    --temperature 0.7 \
    --max_tokens 300

Prepared 1562 batch requests to /workspace/data/qa/llm_batch/output.jsonl
Wrote mapping to /workspace/data/qa/llm_batch/mapping.csv


In [3]:
# Submit LLM batch job and store the output batch ID
from datetime import datetime
import subprocess

# Generate the file path for storing the batch ID
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_id_file = projectRoot / f"BATCH_ID/{timestamp}_batchID"

# Run the command and capture the batch ID

result = subprocess.run(
    [
        "python", "/workspace/src/qa_generation/llm_qa_generator_batch.py", "submit",
        "/workspace/data/qa/llm_batch/output.jsonl",
        "--window", "24h"
    ],
    stdout=subprocess.PIPE,
    text=True,
    check=True
)

# Extract the batch ID from the command output
batch_id = result.stdout.strip()

# Save the batch ID to the file
batch_id_file.parent.mkdir(parents=True, exist_ok=True)
with open(batch_id_file, "w") as file:
    file.write(batch_id)

print(f"Batch ID saved to: {batch_id_file}")

Batch ID saved to: /workspace/BATCH_ID/20260103_231007_batchID


In [14]:
# Check status of LLM batch job
# batch_id = "<batch_id>"  # Replace with the actual batch ID

# Read batch_id from the file
with open(batch_id_file, "r") as file:
    batch_id = file.read().strip()
batch_id = batch_id.split("id=")[1].split(",")[0]

!python /workspace/src/qa_generation/llm_qa_generator_batch.py status \
    {batch_id}

Batch batch_6959a1d67f8c819094e3803ee9c2d338 status: completed
Counts: BatchRequestCounts(completed=1562, failed=0, total=1562)
Output file id: file-QXVvpTXvF9LTtEYSs8wnHG


In [15]:
# Collect LLM batch results
# batch_id = "<batch_id>"  # Replace with the actual batch ID
!python /workspace/src/qa_generation/llm_qa_generator_batch.py collect \
    {batch_id} \
    /workspace/data/qa/llm_batch/output.jsonl \
    /workspace/data/qa/llm_batch/mapping.csv \
    /workspace/data/qa/llm_batch/qa_pairs.csv

Saved batch output JSONL to /workspace/data/qa/llm_batch/output.jsonl
Wrote parsed results to /workspace/data/qa/llm_batch/qa_pairs.csv (1562 rows)
